# **MODELADO**

Este cuaderno parte de `data/03_features/df_final.parquet` (lo escribe `PREPROCESADO_trabajo.ipynb`) y termina en la carpeta `outputs/`, que es lo que lee la aplicación. El modelo es una **regresión logística multinomial** que predice local / empate / visitante (H / D / A) y se entrena **día a día** (walk-forward): para predecir una fecha solo se usan partidos anteriores a esa fecha.

| Apartado | Qué hace | ¿Se ejecuta siempre? |
|---|---|---|
| **0. Preparación** | rutas, datos, la decisión (variables y parámetros) y las funciones comunes | sí |
| **1. Selección de variables** | probar conjuntos, quitar o añadir variables una a una | **solo si se activa** (tarda) |
| **2. El modelo** | walk-forward con las variables elegidas, comparado con Bet365 | sí |
| **3. Predicción** | próxima jornada → `future_predictions_*` | sí |
| **4. Evaluación histórica** | métricas, matriz de confusión, ROC, matchlogs y comparación con Bet365 | sí |
| **5. Radar plot** | datos para la ficha de cada partido → `radar_prematch/` | sí |
| **6. Control final** | comprueba que todos los ficheros están y cuadran entre sí | sí |

**Cómo se usa.** Cada jornada: *Ejecutar todo*. Con las pruebas de variables apagadas tarda alrededor de un minuto. Para hacer pruebas de variables, se pone a `True` el interruptor correspondiente al principio del apartado 1. Las variables del modelo se cambian en **un solo sitio**, el apartado 0.3, y todo lo demás las lee de ahí.

# **0. Preparación**

## 0.1 Dependencias, rutas e importes

In [22]:
# ── DEPENDENCIAS · una sola orden, y la primera celda del cuaderno ───────────
#  Este cuaderno no sale a la red.

!pip install -q pandas numpy pyarrow pytz scikit-learn joblib threadpoolctl jinja2

In [23]:
# =============================================================================
#  CELDA · PARÁMETROS, RUTAS E IMPORTES
# =============================================================================
import hashlib
import json
import os
import re
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pytz
import sklearn
from joblib import Parallel, cpu_count, delayed
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import auc, precision_recall_fscore_support, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

TZ = pytz.timezone("Europe/Madrid")
RUN_DATE = pd.to_datetime(os.environ.get("RUN_DATE") or datetime.now(TZ).date()).strftime("%Y-%m-%d")

ROOT = Path.cwd()
FEAT = ROOT / "data" / "03_features"
MODELS = ROOT / "data" / "04_models"     # resultados de las pruebas de selección
OUT = ROOT / "outputs"                   # lo que lee la aplicación
RADAR = OUT / "radar_prematch"

IN_PATH = FEAT / "df_final.parquet"
if not IN_PATH.exists():
    raise FileNotFoundError(
        f"no existe {IN_PATH}. O el cuaderno no se está ejecutando desde la carpeta del proyecto "
        f"(ahora: {ROOT}), o falta ejecutar PREPROCESADO_trabajo.ipynb, que es quien escribe ese fichero.")

for _p in (MODELS, OUT, RADAR):
    _p.mkdir(parents=True, exist_ok=True)

ETIQUETAS = ["H", "D", "A"]              # orden fijo en todo el cuaderno: local, empate, visitante
EPS = 1e-15
NUCLEOS = max(1, cpu_count() - 2)        # procesos en paralelo para las pruebas de selección

print(f"RUN_DATE = {RUN_DATE} | scikit-learn {sklearn.__version__} | núcleos para las pruebas: {NUCLEOS}")
print(f"ROOT = {ROOT}")

RUN_DATE = 2026-11-25 | scikit-learn 1.4.2 | núcleos para las pruebas: 10
ROOT = c:\Users\admin\Desktop\ultimo proyecto


## 0.2 Datos

In [24]:
# =============================================================================
#  CELDA · CARGA Y CONTROLES DE ENTRADA
# -----------------------------------------------------------------------------
#  Todo lo que viene después da por hecho lo que aquí se comprueba. Si algo
#  falla, el fallo está en el fichero de entrada (PREPROCESADO), no aquí.
# =============================================================================
df = pd.read_parquet(IN_PATH)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Season"] = pd.to_numeric(df["Season"], errors="coerce").astype("Int64")
df["FTR"] = df["FTR"].astype(str).str.upper().str.strip()

_necesarias = ["Date", "Season", "Matchweek", "HomeTeam_norm", "AwayTeam_norm", "FTR",
               "B365H", "B365D", "B365A", "pimp1", "pimpx", "pimp2"]
_faltan = [c for c in _necesarias if c not in df.columns]
if _faltan:
    raise KeyError(f"a {IN_PATH.name} le faltan columnas: {_faltan}")
if df[["Date", "Season", "Matchweek"]].isna().any().any():
    raise ValueError("hay partidos sin fecha, sin temporada o sin jornada en df_final")
_rep = df.duplicated(["Season", "HomeTeam_norm", "AwayTeam_norm"])
if _rep.any():
    raise ValueError("hay partidos repetidos en df_final:\n"
                     + df.loc[_rep, ["Season", "Date", "HomeTeam_norm", "AwayTeam_norm"]].to_string())

# Variables derivadas que se usan en alguno de los conjuntos del apartado 1
df["market_home_logit"] = np.log((df["pimp1"] + 1e-9) / (df["pimp2"] + 1e-9))
df["market_draw_logit"] = np.log((df["pimpx"] + 1e-9) / ((df["pimp1"] + df["pimp2"]) / 2 + 1e-9))
df["elo_diff"] = df["h_elo"] - df["a_elo"]

# Jugado = tiene resultado. Por jugar = no lo tiene (PREPROCESADO lo marca con FTR = "0")
JUGADO = df["FTR"].isin(ETIQUETAS)
TEMPORADA_ACTUAL = int(df["Season"].max())
_fut = df.loc[~JUGADO]

print(f"Leído: {IN_PATH.name} · {len(df):,} partidos · {df.shape[1]} columnas")
print(f"Jugados: {int(JUGADO.sum()):,} (de {df.loc[JUGADO, 'Date'].min():%d/%m/%Y} a {df.loc[JUGADO, 'Date'].max():%d/%m/%Y})")
if _fut.empty:
    print("Por jugar: 0. No hay jornada que predecir: ejecuta antes los cuadernos anteriores.")
else:
    print(f"Por jugar: {len(_fut)} (jornada {sorted(_fut['Matchweek'].astype(int).unique())}, "
          f"de {_fut['Date'].min():%d/%m/%Y} a {_fut['Date'].max():%d/%m/%Y})")
    _pasados = _fut[_fut["Date"] < pd.Timestamp(RUN_DATE)]
    if len(_pasados):
        print(f"\nAVISO · {len(_pasados)} de esos partidos tienen fecha anterior a hoy ({RUN_DATE}) y siguen sin resultado.\n"
              "        O df_final está atrasado (reejecuta extracción, limpieza y preprocesado) o están aplazados.")
    _sin = _fut[_fut[["B365H", "B365D", "B365A"]].isna().any(axis=1)]
    if len(_sin):
        print("\nAVISO · partidos por jugar SIN cuota de Bet365; su predicción no es fiable hasta ponerla:\n"
              + _sin[["Date", "HomeTeam_norm", "AwayTeam_norm"]].to_string(index=False))

Leído: df_final.parquet · 7,739 partidos · 264 columnas
Jugados: 7,729 (de 26/08/2006 a 23/11/2026)
Por jugar: 10 (jornada [14], de 27/11/2026 a 30/11/2026)


## 0.3 La decisión: variables y parámetros del modelo

Este es el **único sitio** donde se decide con qué se predice. La predicción de la próxima jornada, las métricas, los matchlogs y los controles leen de aquí, así que no puede pasar que cada fichero salga de un conjunto distinto.

In [ ]:
# =============================================================================
#  CELDA · LA DECISIÓN
# =============================================================================
FEATURES_FINAL = ['market_home_logit', 'points_venue_last10_diff', 'away_avg_through_balls_last7', 'home_avg_through_balls_last7', 
                 'home_prev_big_odds_win_any', 'nothing_at_stake_diff', 'form_gd_6_diff', 'home_form_gd_6', 'avg_xg_against_last7_diff',
                 'avg_box_touches_last7_diff', 'away_line_move_last5', 'home_avg_big_ch_last7', 'effectiveness_diff', 'away_form_gd_6'] 

WF_KWARGS = dict(
    n_seasons_window=4,     # se entrena con las 4 últimas temporadas...
    season_size=380,        # ...de 380 partidos
    recent_weight=3.0,      # los últimos 380 partidos pesan el triple
    older_weight=1.0,
    C=1.0,                  # regularización de la logística
    max_iter=1000,
)

_faltan = [c for c in FEATURES_FINAL if c not in df.columns]
if _faltan:
    raise KeyError(f"variables de FEATURES_FINAL que no están en df_final: {_faltan}")
if len(set(FEATURES_FINAL)) != len(FEATURES_FINAL):
    raise ValueError("hay variables repetidas en FEATURES_FINAL")
_huecos = [c for c in FEATURES_FINAL if df[c].isna().any()]
if _huecos:
    raise ValueError(f"variables de FEATURES_FINAL con huecos en df_final: {_huecos}")

print(f"{len(FEATURES_FINAL)} variables:", FEATURES_FINAL)
print("Parámetros:", WF_KWARGS)

8 variables: ['pimp1', 'pimpx', 'home_form_gd_6', 'away_relative_perf', 'form_points_6_diff', 'away_avg_big_ch_against_last7', 'home_avg_box_touches_last7', 'home_avg_sofa_rating_last7']
Parámetros: {'n_seasons_window': 4, 'season_size': 380, 'recent_weight': 3.0, 'older_weight': 1.0, 'C': 1.0, 'max_iter': 1000}


## 0.4 Funciones comunes

Una sola versión de cada función, que comparten la selección de variables y el modelo final: así, lo que se mide al elegir es exactamente lo que después se publica.

**Walk-forward.** Para cada fecha con partidos:

1. se toman los **1.520 partidos jugados** inmediatamente anteriores (4 temporadas de 380);
2. los 380 más recientes pesan el triple;
3. se imputan (mediana), se estandarizan y se ajusta la logística, todo **dentro de esa ventana**;
4. se predice la fecha.

Los partidos por jugar se predicen igual, pero **nunca entran en el entrenamiento**.

In [26]:
# =============================================================================
#  CELDA · EL WALK-FORWARD
# =============================================================================
def _norm_name(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).strip().lower())


def pon_claves(d):
    # pred_key = Season|fecha|local|visitante (legible) · pred_key_match = la misma, con los nombres normalizados
    base = d["Season"].astype("Int64").astype(str) + "|" + d["Date"].dt.strftime("%Y-%m-%d") + "|"
    d["pred_key"] = base + d["HomeTeam_norm"].astype(str) + "|" + d["AwayTeam_norm"].astype(str)
    d["pred_key_match"] = base + d["HomeTeam_norm"].map(_norm_name) + "|" + d["AwayTeam_norm"].map(_norm_name)
    return d


META = ["Season", "Date", "Matchweek", "HomeTeam_norm", "AwayTeam_norm", "FTR",
        "B365H", "B365D", "B365A", "pimp1", "pimpx", "pimp2"]


def dias(d, ventana):
    # Recorre las fechas de `d` (ya ordenado) y da, para cada una, las posiciones de los
    # `ventana` últimos partidos jugados antes de ella y las de los partidos que hay que predecir.
    fechas = d["Date"].to_numpy()
    jugados = np.flatnonzero(d["FTR"].isin(ETIQUETAS).to_numpy())
    for fecha in np.unique(fechas):
        entrena = jugados[fechas[jugados] < fecha]
        if entrena.size >= ventana:
            yield entrena[-ventana:], np.flatnonzero(fechas == fecha)


def walkforward(df, feature_cols, n_seasons_window=4, season_size=380, recent_weight=3.0,
                older_weight=1.0, C=1.0, max_iter=1000):
    # Devuelve una fila por partido predicho, con su identificación, sus cuotas, el
    # resultado (y_true), la predicción (y_pred) y las probabilidades (proba_H/D/A).
    feature_cols = list(feature_cols)
    faltan = [c for c in feature_cols if c not in df.columns]
    if faltan:
        raise KeyError(f"variables que no están en los datos: {faltan}")

    # Orden estable: dos partidos de la misma fecha conservan el orden del fichero, y el
    # resultado es el mismo en cualquier ordenador.
    d = df.sort_values("Date", kind="mergesort")
    X, y = d[feature_cols], d["FTR"].to_numpy()

    ventana = n_seasons_window * season_size
    pesos = np.full(ventana, older_weight, dtype=float)
    pesos[-season_size:] = recent_weight

    pipe = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("logit", LogisticRegression(solver="lbfgs", C=C, max_iter=max_iter)),
    ])

    filas, probas = [], []
    with threadpool_limits(limits=1):
        for entrena, predice in dias(d, ventana):
            pipe.fit(X.iloc[entrena], y[entrena], logit__sample_weight=pesos)
            P = pipe.predict_proba(X.iloc[predice])
            clases = list(pipe.named_steps["logit"].classes_)
            if sorted(clases) != sorted(ETIQUETAS):
                raise ValueError(f"{d['Date'].iloc[predice[0]]:%d/%m/%Y}: el entrenamiento tiene las clases {clases}")
            filas.append(predice)
            probas.append(P[:, [clases.index(c) for c in ETIQUETAS]])
    if not filas:
        raise RuntimeError(f"no hay {ventana} partidos jugados antes de ninguna fecha: no se puede predecir nada")

    P = np.vstack(probas)
    r = d.iloc[np.concatenate(filas)][META].rename(columns={"FTR": "y_true"}).reset_index(drop=True)
    r["y_pred"] = np.array(ETIQUETAS)[P.argmax(axis=1)]
    r[["proba_H", "proba_D", "proba_A"]] = P
    r["has_label"] = r["y_true"].isin(ETIQUETAS).astype(int)
    return pon_claves(r)

In [27]:
# =============================================================================
#  CELDA · MÉTRICAS
# -----------------------------------------------------------------------------
#  La apuesta es siempre la misma: 1 unidad al signo predicho, a la cuota de
#  apertura de Bet365, en todos los partidos jugados con cuota válida.
# =============================================================================
CUOTA = {"H": "B365H", "D": "B365D", "A": "B365A"}


def liquida(p, pred_col="y_pred"):
    # Añade a las predicciones: cuota del signo elegido, si se apuesta, si acierta y beneficio.
    r = p.copy()
    r["odds_pick"] = np.select([r[pred_col] == c for c in ETIQUETAS],
                               [pd.to_numeric(r[CUOTA[c]], errors="coerce") for c in ETIQUETAS], np.nan)
    r["bet_placed"] = (r["y_true"].isin(ETIQUETAS) & (r["odds_pick"] >= 1.01)).astype(int)
    r["correct"] = ((r["y_true"] == r[pred_col]) & (r["bet_placed"] == 1)).astype(int)
    r["profit"] = np.where(r["bet_placed"] == 0, np.nan, np.where(r["correct"] == 1, r["odds_pick"] - 1.0, -1.0))
    return r


def log_loss_hda(y, P):
    # P en el orden fijo H, D, A. Se calcula a mano a propósito: `sklearn.log_loss` ordena las
    # clases alfabéticamente (A, D, H) y con las columnas en orden H, D, A daría un número falso.
    col = pd.Series(y).map({c: i for i, c in enumerate(ETIQUETAS)}).to_numpy()
    return float(-np.log(np.clip(np.asarray(P, float)[np.arange(len(col)), col], EPS, 1.0)).mean())


def brier_hda(y, P):
    Y = np.column_stack([(np.asarray(y) == c).astype(float) for c in ETIQUETAS])
    return float(((np.asarray(P, float) - Y) ** 2).sum(axis=1).mean())


def confianza(P):
    # probabilidad máxima, entropía y margen entre la 1ª y la 2ª
    P = np.asarray(P, float)
    o = np.sort(P, axis=1)
    return P.max(axis=1), -(P * np.log(np.clip(P, EPS, 1.0))).sum(axis=1), o[:, -1] - o[:, -2]


def resumen(p, pred_col="y_pred"):
    r = liquida(p, pred_col)
    r = r[r["bet_placed"] == 1]
    return dict(partidos=len(r), acierto=r["correct"].mean(), roi=r["profit"].mean(), beneficio=r["profit"].sum())


def resumen_temporada(p, pred_col="y_pred"):
    r = liquida(p, pred_col)
    r = r[r["bet_placed"] == 1]
    return (r.groupby("Season").agg(bets=("profit", "size"), accuracy=("correct", "mean"),
                                    roi=("profit", "mean"), total_profit=("profit", "sum")).reset_index())


def pon_mercado(p):
    # El modelo de Bet365: predecir siempre el signo con mayor probabilidad implícita.
    r = p.copy()
    M = r[["pimp1", "pimpx", "pimp2"]].to_numpy(float)
    M = M / M.sum(axis=1, keepdims=True)
    r[["pH_mkt_pred", "pD_mkt_pred", "pA_mkt_pred"]] = M
    r["y_pred_market"] = np.array(ETIQUETAS)[M.argmax(axis=1)]
    return r


def ic95(x):
    # media e intervalo del 95 % de una serie de diferencias partido a partido, en puntos
    x = 100 * np.asarray(x, float)
    m, e = x.mean(), 1.96 * x.std(ddof=1) / np.sqrt(len(x))
    return f"{m:+.2f} [{m - e:+.2f}, {m + e:+.2f}]"


def escribe_json(obj, ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, allow_nan=False)

# **1. Selección de variables**

El método es una selección **paso a paso** sobre el walk-forward: se parte de un conjunto, se prueba a añadir (o a quitar) cada variable de una en una, se miden el acierto y el ROI del **periodo elegido**, y con esa información se decide el siguiente conjunto. A diferencia de una *forward selection* clásica, que puntúa sobre los mismos datos con los que entrena, aquí cada partido se predice con un modelo que no lo ha visto.

**El periodo.** `SELECCION_DESDE` y `SELECCION_HASTA` (celda de interruptores) dicen qué temporadas cuentan para elegir. El modelo se entrena igual, con los 1.520 partidos anteriores a cada fecha: lo que cambia es qué partidos puntúan.

* Las temporadas que quedan **fuera** no deciden nada, pero se calculan igual y salen en las columnas `… fuera`: son la comprobación de que una mejora no es solo de esas temporadas.
* El cálculo guardado vale para cualquier periodo: se puede cambiar y volver a ejecutar sin repetir las pruebas.
* El apartado 2 y todo lo que se escribe en `outputs/` no dependen del periodo: siguen teniendo el histórico completo.

Todo este apartado es **opcional** y está apagado por defecto. No escribe nada en `outputs/`: los resultados de las pruebas se guardan en `data/04_models/seleccion/`.

**Cómo leer los resultados.** El modelo no ve los partidos que predice, pero **quien elige la variable sí**: se queda con la que mejor ha funcionado en estas mismas temporadas. Por eso la mejora que se ve aquí es optimista, y cuantas más candidatas se prueban, más fácil es que la ganadora lo sea por azar. Cuatro referencias medidas en este proyecto:

* Añadiendo una columna de **ruido puro** (100 sorteos) a los conjuntos intermedios de esta selección, de 4 a 9 variables, y contando de 2014/15 a 2026/27, el acierto bajó de media entre 0,19 y 0,46 puntos y el ROI entre 0,38 y 1,04. Con unas 250 candidatas, la mejor de puro ruido subiría entre +0,05 y +0,24 de acierto y entre +0,32 y +0,83 de ROI: por debajo de eso, una mejora no se distingue del azar. El listón cambia con el conjunto de partida y con el periodo.
* Lo que se gana en el periodo elegido no tiene por qué aparecer fuera de él: de 5 a 14 variables, el ROI de 2014–2026 subió de +0,42 % a +3,69 %, y el de 2010–2013, que no se usó para elegir, bajó de −2,16 % a −2,95 %. Por eso conviene mirar siempre las columnas `… fuera`.
* Las variables cuyo dato empieza a mitad del histórico (xG en 2014, pases filtrados en 2015, ocasiones claras en 2017, movimiento de la cuota en 2019 y toques en el área en 2023) valen una constante hasta entonces. En sus primeras semanas con dato, la ventana de 1.520 partidos solo tiene unos pocos con valor real y el modelo aprende su peso con ellos: el log loss de ese primer año se dispara y parte de la mejora que se les mide ahí es de ese efecto.
* Lo que aporta cada candidata en 2010–2019 y lo que aporta en 2020–2026 prácticamente no guarda relación (correlación de rangos de +0,01). Conviene mirar siempre si la mejora aparece en las dos mitades.

In [67]:
# =============================================================================
#  CELDA · INTERRUPTORES DE LA SELECCIÓN
# =============================================================================
PROBAR_CONJUNTOS = False        # 1.2 · compara los conjuntos de CONJUNTOS_A_PROBAR
QUITAR_UNA = False              # 1.3 · quita cada variable de BASE_FEATURES, de una en una
ANADIR_UNA = False             # 1.4 · añade cada candidata a BASE_FEATURES, de una en una (la larga)

SELECCION_DESDE = 2014          # primera temporada que cuenta para elegir (2014 = 2014/15) · None = todas
SELECCION_HASTA = None          # última temporada que cuenta · None = hasta la actual

BASE_FEATURES = ['market_home_logit', 'points_venue_last10_diff', 'away_avg_through_balls_last7', 'home_avg_through_balls_last7', 
                 'home_prev_big_odds_win_any', 'nothing_at_stake_diff', 'form_gd_6_diff', 'home_form_gd_6', 'avg_xg_against_last7_diff',
                 'avg_box_touches_last7_diff', 'away_line_move_last5', 'home_avg_big_ch_last7', 'effectiveness_diff', 'away_form_gd_6']                      # conjunto del que se parte en 1.3 y 1.4
CONJUNTOS_A_PROBAR = ["S0", "FINAL"]    # nombres del catálogo de 1.1
TEMPORADA_FOCO = 2026                   # temporada que se mira aparte en las tablas (2025 = 2025/26)
CORTE = 2020                            # las tablas parten el periodo en dos por esta temporada

# -- el periodo, comprobado ---------------------------------------------------
_d = df.sort_values("Date", kind="mergesort")
_prim = int(_d["Season"].iloc[next(dias(_d, WF_KWARGS["n_seasons_window"] * WF_KWARGS["season_size"]))[1][0]])   # primera que se puede predecir
PER_A = max(_prim, SELECCION_DESDE if SELECCION_DESDE is not None else _prim)
PER_Z = min(TEMPORADA_ACTUAL, SELECCION_HASTA if SELECCION_HASTA is not None else TEMPORADA_ACTUAL)
if PER_A > PER_Z:
    raise ValueError(f"periodo de selección vacío: las temporadas que se pueden predecir van de {_prim} a {TEMPORADA_ACTUAL}")


def en_periodo(temporada):
    return (temporada >= PER_A) & (temporada <= PER_Z)


def _tt(a):
    return f"{a}/{(a + 1) % 100:02d}"


PERIODO = f"{_tt(PER_A)} a {_tt(PER_Z)}"
_fuera = [f"{_tt(a)} a {_tt(z)}" if a < z else _tt(a) for a, z in ((_prim, PER_A - 1), (PER_Z + 1, TEMPORADA_ACTUAL)) if a <= z]
print(f"Periodo para elegir: {PERIODO} · fuera, como comprobación: {' y '.join(_fuera) or 'nada'}")

Periodo para elegir: 2014/15 a 2026/27 · fuera, como comprobación: 2010/11 a 2013/14


## 1.1 Catálogo de conjuntos probados

In [29]:
# =============================================================================
#  CELDA · CATÁLOGO DE CONJUNTOS
# -----------------------------------------------------------------------------
#  S0 … S14 son los conjuntos de la primera versión del modelo; N5, N7 y N10,
#  los pasos que llevaron a la elección de 2026/27; V8, las 8 publicadas hasta
#  ahora; FINAL es el que se usa, y se define en la celda 0.3 (LA DECISIÓN).
#  Para probar uno nuevo, se añade aquí y se nombra en CONJUNTOS_A_PROBAR.
# =============================================================================

FEATURES_V8 = [
    "pimp1", "pimpx",                    # probabilidades implícitas de Bet365 (apertura): local y empate
    "home_form_gd_6",                    # diferencia de goles del local en sus 6 últimos partidos
    "away_relative_perf",                # rendimiento del visitante frente a lo que esperaban las cuotas
    "form_points_6_diff",                # puntos en los 6 últimos: local menos visitante
    "away_avg_big_ch_against_last7",     # ocasiones claras que concede el visitante (7 últimos)
    "home_avg_box_touches_last7",        # toques en el área rival del local (7 últimos)
    "home_avg_sofa_rating_last7",        # nota media de SofaScore del local (7 últimos)
]

_precio = ["pimp1", "pimpx", "pimp2"]
_s4 = _precio + ["relative_perf_diff", "avg_xg_last7_diff", "form_points_6_diff", "home_total_gd_cum", "away_total_gd_cum"]
_s6 = _s4 + ["h2h_win_rate_ewm_diff", "home_total_matches_prev", "away_total_matches_prev"]
_s7 = _s6 + ["home_avg_shotsontarget_last7", "avg_shots_last7_diff"]
_h2h = ["h2h_win_rate_ewm_diff", "h2h_draw_rate_ewm_diff", "h2h_loss_rate_ewm_diff"]
_estilo_e = ["home_playstyle_equilibrado", "home_playstyle_ofensivo", "away_playstyle_defensivo", "away_playstyle_ofensivo"]
_estilo_d = ["home_playstyle_defensivo", "home_playstyle_ofensivo", "away_playstyle_defensivo", "away_playstyle_ofensivo"]
_n5 = ["pimp1", "pimpx", "home_form_gd_6", "away_relative_perf", "form_points_6_diff"]
_n7 = _n5 + ["away_avg_big_ch_against_last7", "home_avg_box_touches_last7"]

CONJUNTOS = {
    "S0": _precio,
    "S0p": _precio + ["elo_diff"],
    "S1": _s4[:4],
    "S1p": _s4[:4] + ["elo_diff"],
    "S2": _s4[:5],
    "S2p": _s4[:5] + ["has_xg_data"],
    "S3": _s4[:6],
    "S3p": _s4,
    "S4": _s4 + ["prev_position_diff"],
    "S4p": _s4 + ["home_prev_position", "away_prev_position", "elo_diff"],
    "S5": _s4 + _h2h[:1],
    "S5p": _s4 + _h2h,
    "S6": _s6,
    "S7": _s7,
    "S8": _s4 + _h2h + _estilo_e,
    "S9": _s4 + _h2h + _estilo_e + ["a_elo"],
    "S10": _s4 + _h2h + _estilo_d + ["a_elo", "h2h_draw_rate_roll8_diff"],
    "S11": _s7 + ["away_playstyle_equilibrado"],
    "S11p": _s7 + ["away_gd_cum"],
    "S12": _s7 + ["away_playstyle_equilibrado", "home_prev_big_odds_win_any"],
    "S13": _s7 + ["away_playstyle_equilibrado", "home_prev_big_odds_win_any", "total_gd_cum_diff"],
    "S14": _s7 + ["away_playstyle_equilibrado", "home_prev_big_odds_win_any", "total_gd_cum_diff", "home_avg_xg_last7"],
    "N5": _n5,
    "N7": _n7,
    "N10": _n7 + ["has_sofa_rating_data", "h2h_draw_rate_roll8_diff", "h2h_draw_rate_ewm_diff"],
    "V8": FEATURES_V8,
    "FINAL": FEATURES_FINAL,
}

_rotos = {k: [c for c in v if c not in df.columns] for k, v in CONJUNTOS.items()}
_rotos = {k: v for k, v in _rotos.items() if v}
if _rotos:
    print("AVISO · conjuntos con variables que ya no están en df_final (no se pueden probar):", _rotos)
pd.DataFrame([dict(conjunto=k, variables=len(v), lista=", ".join(v)) for k, v in CONJUNTOS.items()])

,conjunto,variables,lista
0,S0,3,"pimp1, pimpx, pimp2"
1,S0p,4,"pimp1, pimpx, pimp2, elo_diff"
2,S1,4,"pimp1, pimpx, pimp2, relative_perf_diff"
3,S1p,5,"pimp1, pimpx, pimp2, relative_perf_diff, elo_diff"
4,S2,5,"pimp1, pimpx, pimp2, relative_perf_diff, avg_x..."
5,S2p,6,"pimp1, pimpx, pimp2, relative_perf_diff, avg_x..."
6,S3,6,"pimp1, pimpx, pimp2, relative_perf_diff, avg_x..."
7,S3p,8,"pimp1, pimpx, pimp2, relative_perf_diff, avg_x..."
8,S4,9,"pimp1, pimpx, pimp2, relative_perf_diff, avg_x..."
9,S4p,11,"pimp1, pimpx, pimp2, relative_perf_diff, avg_x..."


In [30]:
# =============================================================================
#  CELDA · MOTOR DE LAS PRUEBAS
# -----------------------------------------------------------------------------
#  Cada prueba es un walk-forward completo. Se reparten entre los núcleos y cada
#  resultado se guarda en cuanto llega: si se interrumpe, al volver a ejecutar
#  continúa por donde iba. Lo guardado solo se reutiliza si el fichero de datos,
#  los parámetros y las variables son exactamente los mismos.
# =============================================================================
SELEC = MODELS / "seleccion"


def _firma_datos():
    h = hashlib.md5()
    h.update(IN_PATH.read_bytes())
    h.update(json.dumps(WF_KWARGS, sort_keys=True).encode())
    h.update(sklearn.__version__.encode())
    return h.hexdigest()[:12]


def _una_prueba(datos, cols, kw):
    warnings.simplefilter("ignore")
    r = walkforward(datos, cols, **kw)
    j = r["has_label"] == 1
    ll = {int(s): [log_loss_hda(g["y_true"], g[["proba_H", "proba_D", "proba_A"]]) * len(g), len(g)] for s, g in r[j].groupby("Season")}
    return {"variables": list(cols), "y_pred": "".join(r["y_pred"]), "logloss_temporada": ll}


def prueba_conjuntos(conjuntos):
    # conjuntos: {nombre: lista de variables} → {nombre: predicciones}; el log loss del periodo viaja en .attrs
    carpeta = SELEC / _firma_datos()
    carpeta.mkdir(parents=True, exist_ok=True)
    ruta = {n: carpeta / (hashlib.md5("|".join(c).encode()).hexdigest()[:16] + ".json") for n, c in conjuntos.items()}
    pendientes = [n for n in conjuntos if not ruta[n].exists()]
    print(f"{len(conjuntos)} walk-forwards: {len(conjuntos) - len(pendientes)} ya hechos, {len(pendientes)} por hacer")
    t0 = time.time()
    tareas = (delayed(_una_prueba)(df[list(dict.fromkeys(META + list(conjuntos[n])))], conjuntos[n], WF_KWARGS) for n in pendientes)
    for i, r in enumerate(Parallel(n_jobs=NUCLEOS, return_as="generator")(tareas) if pendientes else [], 1):
        destino = ruta[pendientes[i - 1]]
        tmp = destino.with_suffix(".tmp")
        escribe_json(r, tmp)
        tmp.replace(destino)
        if i % 10 == 0 or i == len(pendientes):
            print(f"   {i}/{len(pendientes)} en {(time.time() - t0) / 60:.1f} min")

    d = df.sort_values("Date", kind="mergesort")
    base = d.iloc[np.concatenate([p for _, p in dias(d, WF_KWARGS["n_seasons_window"] * WF_KWARGS["season_size"])])]
    base = base[META].rename(columns={"FTR": "y_true"}).reset_index(drop=True)
    salida = {}
    for n in conjuntos:
        r = json.loads(ruta[n].read_text(encoding="utf-8"))
        if r["variables"] != list(conjuntos[n]) or len(r["y_pred"]) != len(base):
            raise ValueError(f"lo guardado de '{n}' no corresponde a estos datos: borra {ruta[n]}")
        salida[n] = base.assign(y_pred=list(r["y_pred"]))
        _ll = [v for s, v in r["logloss_temporada"].items() if en_periodo(int(s))]
        salida[n].attrs["logloss"] = sum(v[0] for v in _ll) / sum(v[1] for v in _ll)
    return salida


def tabla_pruebas(res, ref):
    # Cada conjunto, con su diferencia frente a `ref` partido a partido (emparejada). Puntúan los
    # partidos del periodo; las columnas «fuera» son las demás temporadas, que no deciden nada.
    b = liquida(res[ref])
    apuesta = b["bet_placed"] == 1
    ok, fuera = apuesta & en_periodo(b["Season"]), apuesta & ~en_periodo(b["Season"])
    foco = apuesta & (b["Season"] == TEMPORADA_FOCO)
    m1, m2 = ok & (b["Season"] < CORTE), ok & (b["Season"] >= CORTE)
    filas = []
    for n, p in res.items():
        r = liquida(p)
        d_ok, d_roi = r["correct"] - b["correct"], r["profit"] - b["profit"]
        fila = {
            "conjunto": n,
            "acierto": 100 * r.loc[ok, "correct"].mean(), "roi": 100 * r.loc[ok, "profit"].mean(), "logloss": p.attrs["logloss"],
            f"acierto_{TEMPORADA_FOCO}": 100 * r.loc[foco, "correct"].mean(), f"roi_{TEMPORADA_FOCO}": 100 * r.loc[foco, "profit"].mean(),
            "d_acierto": 100 * d_ok[ok].mean(), "d_roi": 100 * d_roi[ok].mean(), "IC 95 % de d_roi": "" if n == ref else ic95(d_roi[ok]).split(" ", 1)[1],
        }
        if m1.any() and m2.any():
            fila |= {f"d_roi hasta {CORTE - 1}": 100 * d_roi[m1].mean(), f"d_roi desde {CORTE}": 100 * d_roi[m2].mean()}
        if fuera.any():
            fila |= {"d_acierto fuera": 100 * d_ok[fuera].mean(), "d_roi fuera": 100 * d_roi[fuera].mean()}
        fila["predicciones distintas"] = int((r["y_pred"] != b["y_pred"])[ok].sum())
        filas.append(fila)
    return pd.DataFrame(filas)


def muestra(t, n=None, orden=None):
    t = t if orden is None else t.sort_values(orden, ascending=False)
    t = (t if n is None else t.head(n)).reset_index(drop=True)
    display(t.style.format({c: "{:.4f}" if c == "logloss" else "{:.2f}" if not c.startswith("d_") else "{:+.2f}"
                            for c in t.select_dtypes("float").columns}))

## 1.2 Probar conjuntos

Compara los conjuntos de `CONJUNTOS_A_PROBAR`. Las columnas `d_…` son la diferencia frente al **primero** de la lista, en puntos. El intervalo se calcula sobre las diferencias partido a partido: si incluye el 0, los dos conjuntos no se distinguen.

In [31]:
if PROBAR_CONJUNTOS:
    res_conjuntos = prueba_conjuntos({n: CONJUNTOS[n] for n in CONJUNTOS_A_PROBAR})
    print(f"Periodo: {PERIODO}")
    muestra(tabla_pruebas(res_conjuntos, CONJUNTOS_A_PROBAR[0]))
    _per = {n: p[en_periodo(p["Season"])] for n, p in res_conjuntos.items()}
    print("ROI por temporada (%):")
    display(pd.DataFrame({n: resumen_temporada(p).set_index("Season")["roi"] * 100 for n, p in _per.items()}).round(2).T)
    print("Acierto por temporada (%):")
    display(pd.DataFrame({n: resumen_temporada(p).set_index("Season")["accuracy"] * 100 for n, p in _per.items()}).round(2).T)
else:
    print("Omitida. Para ejecutarla: PROBAR_CONJUNTOS = True en la celda de interruptores.")

Omitida. Para ejecutarla: PROBAR_CONJUNTOS = True en la celda de interruptores.


## 1.3 Quitar variables, una a una

Responde a qué aporta cada variable del conjunto base: si al quitarla el acierto y el ROI no bajan, sobra.

In [32]:
if QUITAR_UNA:
    _c = {"BASE": list(BASE_FEATURES)}
    _c.update({f"sin {f}": [x for x in BASE_FEATURES if x != f] for f in BASE_FEATURES})
    res_quitar = prueba_conjuntos(_c)
    print(f"Periodo: {PERIODO}")
    muestra(tabla_pruebas(res_quitar, "BASE"))
else:
    print("Omitida. Para ejecutarla: QUITAR_UNA = True en la celda de interruptores.")

Omitida. Para ejecutarla: QUITAR_UNA = True en la celda de interruptores.


## 1.4 Añadir variables, una a una

Prueba cada columna numérica de `df_final` que no esté ya en el conjunto base. Se dejan fuera `Season` y `Matchweek`, que no describen a los equipos, y las columnas constantes, que no pueden cambiar ninguna predicción. Al final salen las 15 mejores por acierto y por ROI, en el periodo y en `TEMPORADA_FOCO`; las columnas `d_…` son la diferencia frente al conjunto base.

Con unas 250 candidatas tarda alrededor de 50 minutos en un equipo de 12 núcleos. Se puede interrumpir: lo ya calculado queda guardado y la siguiente ejecución continúa desde ahí.

In [68]:
if ANADIR_UNA:
    _fuera = set(BASE_FEATURES) | {"Date", "FTR", "HomeTeam_norm", "AwayTeam_norm", "target", "Season", "Matchweek"}
    CANDIDATAS = [c for c in df.columns if c not in _fuera and pd.api.types.is_numeric_dtype(df[c])
                  and df[c].nunique() > 1 and df[c].notna().all()]
    print(f"Candidatas a probar (una a una): {len(CANDIDATAS)}")
    if BASE_FEATURES:
        _ref = "Conjunto base"
        _c = {"BASE": list(BASE_FEATURES)}
        _c.update({f: list(BASE_FEATURES) + [f] for f in CANDIDATAS})
        _res = prueba_conjuntos(_c)
    else:
        # sin variables de partida: cada candidata sola, comparada con apostar siempre al favorito de Bet365
        _ref = "Referencia, siempre al favorito de Bet365"
        _solas = prueba_conjuntos({f: [f] for f in CANDIDATAS})
        _m = pon_mercado(next(iter(_solas.values())))
        _m["y_pred"] = _m["y_pred_market"]
        _j = _m["y_true"].isin(ETIQUETAS) & en_periodo(_m["Season"])
        _m.attrs["logloss"] = log_loss_hda(_m.loc[_j, "y_true"], _m.loc[_j, ["pH_mkt_pred", "pD_mkt_pred", "pA_mkt_pred"]])
        _res = {"BASE": _m, **_solas}
    resultados = tabla_pruebas(_res, "BASE")
    _b = resultados.iloc[0]
    for _m in ("acierto", "roi"):
        resultados[f"d_{_m}_{TEMPORADA_FOCO}"] = resultados[f"{_m}_{TEMPORADA_FOCO}"] - _b[f"{_m}_{TEMPORADA_FOCO}"]
    print(f"\nPeriodo: {PERIODO}\n{_ref}: acierto {_b['acierto']:.2f} % · ROI {_b['roi']:+.2f} % · "
          f"{TEMPORADA_FOCO}: acierto {_b[f'acierto_{TEMPORADA_FOCO}']:.2f} % · ROI {_b[f'roi_{TEMPORADA_FOCO}']:+.2f} %")
    _r = resultados.iloc[1:].rename(columns={"conjunto": "variable"})
    for _col, _tit in (("acierto", "acierto en el periodo"), ("roi", "ROI en el periodo"),
                       (f"acierto_{TEMPORADA_FOCO}", f"acierto {TEMPORADA_FOCO}"), (f"roi_{TEMPORADA_FOCO}", f"ROI {TEMPORADA_FOCO}")):
        print(f"\n== TOP 15 por {_tit} ==")
        muestra(_r, 15, _col)
    _csv = SELEC / f"anadir_una_{RUN_DATE}_{PER_A}-{PER_Z}_desde_{len(BASE_FEATURES)}.csv"
    resultados.to_csv(_csv, index=False)
    print(f"Tabla completa en {_csv}")
else:
    print("Omitida. Para ejecutarla: ANADIR_UNA = True en la celda de interruptores.")

Candidatas a probar (una a una): 240
241 walk-forwards: 1 ya hechos, 240 por hacer
   10/240 en 3.9 min
   20/240 en 6.6 min
   30/240 en 9.4 min
   40/240 en 12.2 min
   50/240 en 15.0 min
   60/240 en 17.7 min
   70/240 en 20.4 min
   80/240 en 23.2 min
   90/240 en 26.1 min
   100/240 en 28.7 min
   110/240 en 31.4 min
   120/240 en 34.3 min
   130/240 en 37.1 min
   140/240 en 39.9 min
   150/240 en 42.7 min
   160/240 en 45.5 min
   170/240 en 48.8 min
   180/240 en 51.5 min
   190/240 en 54.5 min
   200/240 en 57.3 min
   210/240 en 60.0 min
   220/240 en 62.9 min
   230/240 en 65.8 min
   240/240 en 68.2 min

Periodo: 2014/15 a 2026/27
Conjunto base: acierto 55.34 % · ROI +3.69 % · 2026: acierto 59.69 % · ROI +12.73 %

== TOP 15 por acierto en el periodo ==


,variable,acierto,roi,logloss,acierto_2026,roi_2026,d_acierto,d_roi,IC 95 % de d_roi,d_roi hasta 2019,d_roi desde 2020,d_acierto fuera,d_roi fuera,predicciones distintas,d_acierto_2026,d_roi_2026
0,has_tracking_data,55.32,3.52,1.0031,59.69,12.73,-0.02,-0.16,"[-0.75, +0.42]",+0.00,-0.32,+0.00,+0.00,36,+0.00,+0.00
1,avg_through_balls_last7_diff,55.32,3.64,1.0027,59.69,12.73,-0.02,-0.04,"[-0.13, +0.04]",+0.00,-0.09,+0.00,+0.00,1,+0.00,+0.00
2,has_box_touches_data,55.30,3.53,1.0055,59.69,12.73,-0.04,-0.15,"[-0.71, +0.40]",+0.00,-0.30,+0.00,+0.00,30,+0.00,+0.00
3,away_prev_big_odds_loss_any,55.28,3.59,1.0036,58.91,10.75,-0.06,-0.10,"[-0.74, +0.55]",-0.13,-0.06,+0.00,+0.06,43,-0.78,-1.98
4,avg_distance_km_last7_diff,55.24,3.55,1.0060,58.91,11.53,-0.11,-0.14,"[-0.75, +0.48]",+0.00,-0.27,+0.00,+0.00,43,-0.78,-1.20
5,away_prev_big_odds_win_any,55.21,3.42,1.0039,59.69,12.73,-0.13,-0.27,"[-0.94, +0.40]",-0.23,-0.31,-0.07,-0.18,43,+0.00,+0.00
6,prev_big_odds_win_any_diff,55.21,3.42,1.0032,59.69,12.73,-0.13,-0.27,"[-0.94, +0.40]",-0.23,-0.31,-0.07,-0.18,42,+0.00,+0.00
7,home_avg_box_touches_against_last7,55.19,3.40,1.0099,58.91,11.72,-0.15,-0.29,"[-1.08, +0.51]",+0.00,-0.56,+0.00,+0.00,70,-0.78,-1.01
8,away_avg_distance_km_last7,55.19,3.44,1.0046,60.47,14.28,-0.15,-0.25,"[-0.95, +0.45]",+0.00,-0.49,+0.00,+0.00,45,+0.78,+1.55
9,europe_fight_diff,55.19,3.34,1.0036,58.91,10.75,-0.15,-0.35,"[-1.05, +0.35]",-0.28,-0.42,-0.26,-0.58,57,-0.78,-1.98



== TOP 15 por ROI en el periodo ==


,variable,acierto,roi,logloss,acierto_2026,roi_2026,d_acierto,d_roi,IC 95 % de d_roi,d_roi hasta 2019,d_roi desde 2020,d_acierto fuera,d_roi fuera,predicciones distintas,d_acierto_2026,d_roi_2026
0,avg_through_balls_last7_diff,55.32,3.64,1.0027,59.69,12.73,-0.02,-0.04,"[-0.13, +0.04]",+0.00,-0.09,+0.00,+0.00,1,+0.00,+0.00
1,away_prev_big_odds_loss_any,55.28,3.59,1.0036,58.91,10.75,-0.06,-0.10,"[-0.74, +0.55]",-0.13,-0.06,+0.00,+0.06,43,-0.78,-1.98
2,avg_distance_km_last7_diff,55.24,3.55,1.0060,58.91,11.53,-0.11,-0.14,"[-0.75, +0.48]",+0.00,-0.27,+0.00,+0.00,43,-0.78,-1.20
3,has_box_touches_data,55.30,3.53,1.0055,59.69,12.73,-0.04,-0.15,"[-0.71, +0.40]",+0.00,-0.30,+0.00,+0.00,30,+0.00,+0.00
4,has_tracking_data,55.32,3.52,1.0031,59.69,12.73,-0.02,-0.16,"[-0.75, +0.42]",+0.00,-0.32,+0.00,+0.00,36,+0.00,+0.00
5,away_avg_distance_km_last7,55.19,3.44,1.0046,60.47,14.28,-0.15,-0.25,"[-0.95, +0.45]",+0.00,-0.49,+0.00,+0.00,45,+0.78,+1.55
6,prev_big_odds_win_any_diff,55.21,3.42,1.0032,59.69,12.73,-0.13,-0.27,"[-0.94, +0.40]",-0.23,-0.31,-0.07,-0.18,42,+0.00,+0.00
7,away_prev_big_odds_win_any,55.21,3.42,1.0039,59.69,12.73,-0.13,-0.27,"[-0.94, +0.40]",-0.23,-0.31,-0.07,-0.18,43,+0.00,+0.00
8,home_avg_box_touches_against_last7,55.19,3.40,1.0099,58.91,11.72,-0.15,-0.29,"[-1.08, +0.51]",+0.00,-0.56,+0.00,+0.00,70,-0.78,-1.01
9,europe_fight_diff,55.19,3.34,1.0036,58.91,10.75,-0.15,-0.35,"[-1.05, +0.35]",-0.28,-0.42,-0.26,-0.58,57,-0.78,-1.98



== TOP 15 por acierto 2026 ==


,variable,acierto,roi,logloss,acierto_2026,roi_2026,d_acierto,d_roi,IC 95 % de d_roi,d_roi hasta 2019,d_roi desde 2020,d_acierto fuera,d_roi fuera,predicciones distintas,d_acierto_2026,d_roi_2026
0,home_avg_xgot_against_last7,54.87,2.74,1.0043,61.24,17.77,-0.47,-0.94,"[-1.78, -0.11]",+0.00,-1.84,+0.00,+0.00,68,+1.55,+5.04
1,B365H,54.57,1.99,1.0026,61.24,17.77,-0.77,-1.70,"[-2.68, -0.72]",-2.14,-1.28,+0.07,+0.29,106,+1.55,+5.04
2,away_relative_perf,54.98,2.89,1.0035,61.24,17.77,-0.36,-0.80,"[-1.71, +0.11]",-0.91,-0.69,+0.20,+0.68,81,+1.55,+5.04
3,home_final_position_prev_season,54.77,2.66,1.0019,61.24,17.22,-0.58,-1.03,"[-2.38, +0.32]",-0.30,-1.72,+0.00,+0.30,186,+1.55,+4.50
4,a_elo_esc,54.83,2.59,1.0031,61.24,17.92,-0.51,-1.10,"[-2.16, -0.05]",-1.68,-0.55,-0.33,-0.70,112,+1.55,+5.19
5,pct_foreigners_prev_season_diff,54.64,2.11,1.0036,61.24,17.77,-0.70,-1.58,"[-2.51, -0.65]",-1.01,-2.12,+0.39,+1.22,101,+1.55,+5.04
6,away_elo_change_last10,54.83,2.62,1.0027,61.24,17.77,-0.51,-1.07,"[-2.15, +0.01]",-0.43,-1.67,-0.13,-0.11,119,+1.55,+5.04
7,line_move_last5_diff,55.07,3.02,1.0069,60.47,15.25,-0.28,-0.67,"[-1.52, +0.18]",-0.07,-1.23,+0.00,+0.00,78,+0.78,+2.52
8,home_line_move_last5,55.04,2.99,1.0081,60.47,15.25,-0.30,-0.70,"[-1.55, +0.15]",-0.14,-1.23,+0.00,+0.00,78,+0.78,+2.52
9,h_pct_foreigners_prev_season,54.66,2.11,1.0042,60.47,15.25,-0.68,-1.58,"[-2.52, -0.65]",-1.43,-1.73,+0.53,+1.52,100,+0.78,+2.52



== TOP 15 por ROI 2026 ==


,variable,acierto,roi,logloss,acierto_2026,roi_2026,d_acierto,d_roi,IC 95 % de d_roi,d_roi hasta 2019,d_roi desde 2020,d_acierto fuera,d_roi fuera,predicciones distintas,d_acierto_2026,d_roi_2026
0,a_elo_esc,54.83,2.59,1.0031,61.24,17.92,-0.51,-1.10,"[-2.16, -0.05]",-1.68,-0.55,-0.33,-0.70,112,+1.55,+5.19
1,B365H,54.57,1.99,1.0026,61.24,17.77,-0.77,-1.70,"[-2.68, -0.72]",-2.14,-1.28,+0.07,+0.29,106,+1.55,+5.04
2,away_relative_perf,54.98,2.89,1.0035,61.24,17.77,-0.36,-0.80,"[-1.71, +0.11]",-0.91,-0.69,+0.20,+0.68,81,+1.55,+5.04
3,away_elo_change_last10,54.83,2.62,1.0027,61.24,17.77,-0.51,-1.07,"[-2.15, +0.01]",-0.43,-1.67,-0.13,-0.11,119,+1.55,+5.04
4,pct_foreigners_prev_season_diff,54.64,2.11,1.0036,61.24,17.77,-0.70,-1.58,"[-2.51, -0.65]",-1.01,-2.12,+0.39,+1.22,101,+1.55,+5.04
5,home_avg_xgot_against_last7,54.87,2.74,1.0043,61.24,17.77,-0.47,-0.94,"[-1.78, -0.11]",+0.00,-1.84,+0.00,+0.00,68,+1.55,+5.04
6,home_final_position_prev_season,54.77,2.66,1.0019,61.24,17.22,-0.58,-1.03,"[-2.38, +0.32]",-0.30,-1.72,+0.00,+0.30,186,+1.55,+4.50
7,home_avg_sofa_rating_last7,55.15,3.29,1.0096,60.47,16.91,-0.19,-0.40,"[-0.85, +0.05]",+0.00,-0.78,+0.00,+0.00,25,+0.78,+4.19
8,a_value_mio_prev_season,54.75,2.47,1.0030,60.47,15.95,-0.60,-1.22,"[-2.34, -0.11]",-0.60,-1.81,+0.26,+0.85,129,+0.78,+3.22
9,away_h2h_win_rate_ewm,54.51,1.73,1.0027,60.47,15.91,-0.83,-1.96,"[-3.05, -0.87]",-2.19,-1.74,-0.07,-0.04,130,+0.78,+3.18


Tabla completa en c:\Users\admin\Desktop\ultimo proyecto\data\04_models\seleccion\anadir_una_2026-11-25_2014-2026_desde_14.csv


# **2. El modelo**

Walk-forward con las variables de `FEATURES_FINAL`. De aquí sale `preds`, la tabla con la predicción de cada partido de la que se alimentan **todos** los apartados siguientes, incluida la próxima jornada.

In [13]:
# =============================================================================
#  CELDA · EL MODELO
# =============================================================================
_t = time.time()
preds = pon_mercado(walkforward(df, FEATURES_FINAL, **WF_KWARGS))
JUG = preds["has_label"] == 1

# -- controles: si algo de esto falla, nada de lo que sigue valdría ----------
_P = preds[["proba_H", "proba_D", "proba_A"]].to_numpy()
if not np.allclose(_P.sum(axis=1), 1.0, atol=1e-9):
    raise ValueError("las probabilidades de algún partido no suman 1")
if preds["pred_key"].duplicated().any():
    raise ValueError("hay partidos repetidos en las predicciones")
if int((~JUG).sum()) != int((~JUGADO).sum()):
    raise ValueError("no se han predicho todos los partidos por jugar")
_sin_cuota = int(preds.loc[JUG, ["B365H", "B365D", "B365A"]].isna().any(axis=1).sum())
_margen = (1 / preds[["B365H", "B365D", "B365A"]]).sum(axis=1, skipna=False)
_raro = int(((_margen < 1.0) | (_margen > 1.3)).sum())

print(f"{int(JUG.sum()):,} partidos evaluados ({preds['Season'].min()}–{preds['Season'].max()}) "
      f"+ {int((~JUG).sum())} por jugar, en {time.time() - _t:.0f} s")
print(f"[CHECK] jugados sin cuota: {_sin_cuota} · margen de la casa fuera de [1,00 ; 1,30]: {_raro}")

RES_MODELO, RES_BET365 = resumen(preds), resumen(preds, "y_pred_market")
display(pd.DataFrame({
    "modelo": [f"{100 * RES_MODELO['acierto']:.2f} %", f"{100 * RES_MODELO['roi']:+.2f} %", f"{RES_MODELO['beneficio']:+.2f}", RES_MODELO["partidos"]],
    "Bet365 (favorito)": [f"{100 * RES_BET365['acierto']:.2f} %", f"{100 * RES_BET365['roi']:+.2f} %", f"{RES_BET365['beneficio']:+.2f}", RES_BET365["partidos"]],
}, index=["acierto", "ROI", "beneficio (unidades)", "apuestas"]))
_dif = liquida(preds)["profit"] - liquida(preds, "y_pred_market")["profit"]
print(f"Ventaja de ROI sobre Bet365, partido a partido: {ic95(_dif.dropna())} puntos · "
      f"el modelo discrepa del favorito en {100 * (preds['y_pred'] != preds['y_pred_market'])[JUG].mean():.1f} % de los partidos")

if not en_periodo(preds.loc[JUG, "Season"]).all():
    _pp = preds[en_periodo(preds["Season"])]
    _a, _b = resumen(_pp), resumen(_pp, "y_pred_market")
    _dif = (liquida(_pp)["profit"] - liquida(_pp, "y_pred_market")["profit"]).dropna()
    print(f"En el periodo {PERIODO} ({_a['partidos']:,} partidos): modelo {100 * _a['acierto']:.2f} % y ROI {100 * _a['roi']:+.2f} % ({_a['beneficio']:+.2f} u) · "
          f"Bet365 {100 * _b['acierto']:.2f} % y {100 * _b['roi']:+.2f} % ({_b['beneficio']:+.2f} u) · ventaja {ic95(_dif)}")

print("\nPor temporada:")
_t1, _t2 = resumen_temporada(preds), resumen_temporada(preds, "y_pred_market")
display(_t1.merge(_t2, on="Season", suffixes=("_modelo", "_bet365")).drop(columns="bets_bet365")
        .rename(columns={"bets_modelo": "partidos"})
        .style.format({"accuracy_modelo": "{:.2%}", "accuracy_bet365": "{:.2%}", "roi_modelo": "{:+.2%}", "roi_bet365": "{:+.2%}",
                       "total_profit_modelo": "{:+.2f}", "total_profit_bet365": "{:+.2f}"}).hide(axis="index"))

6,209 partidos evaluados (2010–2026) + 10 por jugar, en 51 s
[CHECK] jugados sin cuota: 0 · margen de la casa fuera de [1,00 ; 1,30]: 0


,modelo,Bet365 (favorito)
acierto,55.19 %,54.58 %
ROI,+0.01 %,-2.77 %
beneficio (unidades),+0.72,-171.94
apuestas,6209,6209


Ventaja de ROI sobre Bet365, partido a partido: +2.78 [+1.25, +4.31] puntos · el modelo discrepa del favorito en 6.8 % de los partidos

Por temporada:


Season,partidos,accuracy_modelo,roi_modelo,total_profit_modelo,accuracy_bet365,roi_bet365,total_profit_bet365
2010,380,61.58%,+11.31%,+42.99,61.05%,+9.73%,+36.99
2011,380,54.21%,-6.29%,-23.89,53.68%,-7.91%,-30.06
2012,380,55.00%,-4.22%,-16.05,54.74%,-5.11%,-19.43
2013,380,55.26%,-5.24%,-19.91,53.95%,-9.22%,-35.03
2014,380,56.05%,-6.94%,-26.37,56.84%,-5.30%,-20.14
2015,380,55.53%,-3.08%,-11.71,54.74%,-6.26%,-23.78
2016,380,60.00%,+4.47%,+16.97,58.42%,-0.67%,-2.53
2017,380,55.79%,+0.47%,+1.78,54.74%,-3.60%,-13.69
2018,380,48.95%,-10.93%,-41.53,48.16%,-14.12%,-53.67
2019,380,51.84%,-3.25%,-12.35,51.84%,-4.06%,-15.43


# **3. Predicción de la próxima jornada**

Escribe `future_predictions_<temporada>.csv` y `.json`, y un resumen `future_predictions_summary_<fecha>-<código>.json` que guarda las variables y los parámetros con los que se predijo. El código se calcula a partir de las propias predicciones: si son las mismas, el código es el mismo.

In [14]:
# =============================================================================
#  CELDA · PREDICCIONES FUTURAS → outputs/future_predictions_*
# =============================================================================
fut = preds[~JUG].sort_values(["Date", "HomeTeam_norm", "AwayTeam_norm"], kind="mergesort").reset_index(drop=True)
RES_FUTURO = None

if fut.empty:
    print("No hay partidos por jugar en df_final: no se escribe nada.\n"
          "AVISO · los future_predictions_* que haya en outputs/ son de una jornada anterior.")
else:
    fut = fut.rename(columns={"Matchweek": "Matchday", "proba_H": "pH_pred", "proba_D": "pD_pred", "proba_A": "pA_pred"})
    fut["conf_maxprob"], fut["entropy"], fut["margin_top12"] = confianza(fut[["pH_pred", "pD_pred", "pA_pred"]])
    _cols = ["Season", "Date", "Matchday", "HomeTeam_norm", "AwayTeam_norm", "pred_key", "B365H", "B365D", "B365A",
             "y_pred", "pH_pred", "pD_pred", "pA_pred", "conf_maxprob", "entropy", "margin_top12"]

    creados = []
    for s, g in fut.groupby("Season"):
        _csv, _json = OUT / f"future_predictions_{int(s)}.csv", OUT / f"future_predictions_{int(s)}.json"
        g[_cols].to_csv(_csv, index=False)
        g[_cols].to_json(_json, orient="records", date_format="iso")
        creados.append({"season": int(s), "csv": f"outputs/{_csv.name}", "json": f"outputs/{_json.name}", "n_rows": len(g)})

    _d = fut[["Season", "Date", "HomeTeam_norm", "AwayTeam_norm", "y_pred", "pH_pred", "pD_pred", "pA_pred"]].copy()
    _d["Date"] = _d["Date"].dt.strftime("%Y-%m-%d")
    _d[["pH_pred", "pD_pred", "pA_pred"]] = _d[["pH_pred", "pD_pred", "pA_pred"]].round(10)
    _d = _d.sort_values(["Season", "Date", "HomeTeam_norm", "AwayTeam_norm"], kind="mergesort")
    run_code = f"{int(hashlib.md5(_d.to_csv(index=False, lineterminator=chr(10)).encode('utf-8')).hexdigest()[:12], 16) % 10**6:06d}"

    _dia = fut.assign(date=fut["Date"].dt.strftime("%Y-%m-%d")).groupby("date").agg(
        n_matches=("pred_key", "count"), mean_conf=("conf_maxprob", "mean"),
        mean_entropy=("entropy", "mean"), mean_margin=("margin_top12", "mean"),
        pct_pick_H=("y_pred", lambda x: float((x == "H").mean())),
        pct_pick_D=("y_pred", lambda x: float((x == "D").mean())),
        pct_pick_A=("y_pred", lambda x: float((x == "A").mean()))).reset_index()

    _sum = OUT / f"future_predictions_summary_{RUN_DATE.replace('-', '')}-{run_code}.json"
    escribe_json({
        "generated_at": RUN_DATE.replace("-", ""), "run_code": run_code,
        "model": {"type": "LogisticRegression(multinomial)", **WF_KWARGS, "features": list(FEATURES_FINAL),
                  "classes_order": sorted(ETIQUETAS),
                  "proba_mapping": {"pH_pred": "H", "pD_pred": "D", "pA_pred": "A"}},
        "filters": {"season_filter": TEMPORADA_ACTUAL},
        "data": {"n_future_rows_out": len(fut), "future_min_date": str(fut["Date"].min()), "future_max_date": str(fut["Date"].max()),
                 "unique_key_report": {"collisions_augmented": 0}, "per_season_exports": creados},
        "by_date": _dia.to_dict(orient="records"),
    }, _sum)
    RES_FUTURO = {"created": creados, "summary_path": f"outputs/{_sum.name}", "total_rows": len(fut)}

    display(fut[["Date", "Matchday", "HomeTeam_norm", "AwayTeam_norm", "B365H", "B365D", "B365A", "y_pred", "pH_pred", "pD_pred", "pA_pred"]]
            .style.format({"Date": "{:%d/%m/%Y}", "B365H": "{:.2f}", "B365D": "{:.2f}", "B365A": "{:.2f}",
                           "pH_pred": "{:.1%}", "pD_pred": "{:.1%}", "pA_pred": "{:.1%}"}).hide(axis="index"))
    print(RES_FUTURO)

Date,Matchday,HomeTeam_norm,AwayTeam_norm,B365H,B365D,B365A,y_pred,pH_pred,pD_pred,pA_pred
27/11/2026,14,levante,santander,2.05,3.60,3.20,H,38.7%,33.1%,28.3%
28/11/2026,14,betis,vallecano,1.57,4.10,5.25,H,57.3%,20.7%,22.0%
28/11/2026,14,elche,ath madrid,6.25,4.33,1.44,A,7.9%,15.1%,77.0%
28/11/2026,14,malaga,ath bilbao,4.20,3.75,1.72,A,20.5%,22.3%,57.2%
28/11/2026,14,real madrid,alaves,1.12,8.00,17.00,H,80.0%,6.6%,13.5%
29/11/2026,14,celta,villarreal,2.70,3.40,2.45,A,26.0%,32.2%,41.8%
29/11/2026,14,la coruna,barcelona,9.00,5.75,1.25,A,7.5%,10.5%,82.0%
29/11/2026,14,sociedad,sevilla,2.10,3.50,3.20,H,38.9%,22.3%,38.9%
29/11/2026,14,valencia,osasuna,2.10,3.30,3.25,D,33.2%,36.2%,30.6%
30/11/2026,14,espanyol,getafe,1.95,3.25,4.10,D,37.1%,37.2%,25.7%


{'created': [{'season': 2026, 'csv': 'outputs/future_predictions_2026.csv', 'json': 'outputs/future_predictions_2026.json', 'n_rows': 10}], 'summary_path': 'outputs/future_predictions_summary_20261125-002933.json', 'total_rows': 10}


# **4. Evaluación histórica**

## 4.1 Métricas principales

La primera métrica es el **Accuracy** (porcentaje de aciertos totales), pero esta métrica ignora cómo de seguras son esas predicciones.

$$
\text{Accuracy} = \frac{\text{Número de aciertos}}{\text{Número total de predicciones}}
$$

Para ello se utiliza el **Log Loss** (Cross-Entropy Loss), métrica que mide qué tan buenas son las probabilidades que predice mi modelo de clasificación. A esta métrica no solo le importa acertar la clase, sino cuán seguro está el modelo.

$$
\text{LogLoss} = -\frac{1}{N} \sum_{i=1}^{N} \sum_{j=1}^{K} y_{ij} \cdot \log(p_{ij})
$$

donde:

- $y_{ij}$ = 1 si la clase real del ejemplo $i$ es la clase $j$, y 0 en caso contrario.
- $p_{ij}$ es la probabilidad predicha por el modelo de que el ejemplo $i$ pertenezca a la clase $j$.

Tener un Log Loss alto en este caso significaría dar una probabilidad alta a la clase incorrecta, o lo que es lo mismo, dar una probabilidad baja a la clase correcta.

Por último añadí también el **Brier Score**, que es una métrica que evalúa cuán cercanas están las probabilidades predichas por el modelo respecto a la realidad, comparando la distribución de probabilidades contra la clase real (codificada en one-hot). Es como un error cuadrático medio (MSE) para probabilidades.

$$
\text{Brier Score} = \frac{1}{N} \sum_{i=1}^{N} \sum_{j=1}^{K} (p_{ij} - y_{ij})^2
$$

donde:

- $N$ es el número de ejemplos.
- $K$ es el número de clases (en este caso 3: victoria local, empate, victoria visitante).
- $p_{ij}$ es la probabilidad predicha por el modelo de que el ejemplo $i$ pertenezca a la clase $j$.
- $y_{ij}$ es 1 si la clase real del ejemplo $i$ es la clase $j$, y 0 en caso contrario.

Un Brier Score de 0 significa que las probabilidades dadas por el modelo son perfectas, mientras que uno de 0,67 en nuestro caso sería el de un modelo que reparte la probabilidad a partes iguales entre los tres resultados.

In [15]:
# =============================================================================
#  CELDA · MÉTRICAS POR TEMPORADA → outputs/metrics_main_by_season.csv
# =============================================================================
def metricas_por_temporada(p, pred_col, proba_cols):
    r = liquida(p[p["has_label"] == 1], pred_col)
    r["overround"] = (1 / r[["B365H", "B365D", "B365A"]].clip(lower=1.0)).sum(axis=1, skipna=False)
    r["conf"], r["ent"], r["mar"] = confianza(r[proba_cols])
    filas = []
    for s, g in r.groupby("Season"):
        a = g[g["bet_placed"] == 1]
        filas.append({
            "Season": int(s), "accuracy": (g["y_true"] == g[pred_col]).mean(),
            "logloss": log_loss_hda(g["y_true"], g[proba_cols]), "brier": brier_hda(g["y_true"], g[proba_cols]), "n_scored": len(g),
            "roi": a["profit"].mean(), "n_bets": len(a), "n_wins": int(a["correct"].sum()), "hit_rate": a["correct"].mean(),
            "avg_odds_win": a.loc[a["correct"] == 1, "odds_pick"].mean(), "avg_overround": a["overround"].mean(),
            "avg_conf": g["conf"].mean(), "avg_entropy": g["ent"].mean(), "avg_margin": g["mar"].mean(),
        })
    return pd.DataFrame(filas), r


PROBA = ["proba_H", "proba_D", "proba_A"]
metrics_main, _ = metricas_por_temporada(preds, "y_pred", PROBA)
metrics_main = metrics_main[["Season", "accuracy", "logloss", "brier", "roi", "n_bets", "n_wins", "hit_rate", "avg_odds_win",
                             "avg_overround", "avg_conf", "avg_entropy", "avg_margin"]]
metrics_main.to_csv(OUT / "metrics_main_by_season.csv", index=False)
print("✔ outputs/metrics_main_by_season.csv")
display(metrics_main.round(4))

✔ outputs/metrics_main_by_season.csv


,Season,accuracy,logloss,brier,roi,n_bets,n_wins,hit_rate,avg_odds_win,avg_overround,avg_conf,avg_entropy,avg_margin
0,2010,0.6158,0.9240,0.5397,0.1131,380,234,0.6158,1.8076,1.0657,0.5754,0.9269,0.3344
1,2011,0.5421,0.9478,0.5635,-0.0629,380,206,0.5421,1.7287,1.0645,0.5775,0.9191,0.3381
2,2012,0.5500,0.9557,0.5651,-0.0422,380,209,0.5500,1.7414,1.0637,0.5817,0.9224,0.3486
3,2013,0.5526,0.9568,0.5666,-0.0524,380,210,0.5526,1.7147,1.0636,0.5791,0.9051,0.3507
4,2014,0.5605,0.9133,0.5368,-0.0694,380,213,0.5605,1.6602,1.0551,0.5641,0.9225,0.3286
5,2015,0.5553,0.9323,0.5514,-0.0308,380,211,0.5553,1.7455,1.0512,0.5573,0.9278,0.3108
6,2016,0.6000,0.9070,0.5343,0.0447,380,228,0.6000,1.7411,1.0508,0.5698,0.9138,0.3295
7,2017,0.5579,1.0484,0.5865,0.0047,380,212,0.5579,1.8008,1.0527,0.5698,0.9208,0.3273
8,2018,0.4895,1.0211,0.6103,-0.1093,380,186,0.4895,1.8197,1.0526,0.5217,0.9790,0.2525
9,2019,0.5184,0.9913,0.5926,-0.0325,380,197,0.5184,1.8662,1.0547,0.5168,0.9831,0.2419


## 4.2 Matriz de confusión

In [16]:
# =============================================================================
#  CELDA · MATRICES DE CONFUSIÓN → outputs/confusion_matrices_by_season.json
#  Filas = resultado real, columnas = predicción, en el orden H, D, A.
# =============================================================================
def matriz(g):
    M = pd.crosstab(g["y_true"], g["y_pred"]).reindex(index=ETIQUETAS, columns=ETIQUETAS, fill_value=0).astype(int)
    return {"labels": ETIQUETAS, "matrix": M.to_numpy().tolist(),
            "support": {c: int((g["y_true"] == c).sum()) for c in ETIQUETAS}, "n_scored": len(g)}


_e = preds[JUG]
confusion = {
    "meta": {"row_axis": "y_true", "col_axis": "y_pred", "labels_order": ETIQUETAS},
    "by_season": [{"Season": int(s), **matriz(g)} for s, g in _e.groupby("Season")],
    "overall": matriz(_e),
}
escribe_json(confusion, OUT / "confusion_matrices_by_season.json")
print("✔ outputs/confusion_matrices_by_season.json · total de temporadas:")
display(pd.DataFrame(confusion["overall"]["matrix"], index=[f"real {c}" for c in ETIQUETAS], columns=[f"predicho {c}" for c in ETIQUETAS]))

✔ outputs/confusion_matrices_by_season.json · total de temporadas:


,predicho H,predicho D,predicho A
real H,2441,73,379
real D,1014,81,460
real A,793,63,905


## 4.3 Métricas de clasificación

In [17]:
# =============================================================================
#  CELDA · PRECISIÓN, RECALL Y F1 → outputs/classification_report_by_season.csv
# =============================================================================
_filas = []
for s, g in preds[JUG].groupby("Season"):
    fila = {"Season": int(s)}
    for tipo in ("macro", "weighted"):
        p_, r_, f_, _ = precision_recall_fscore_support(g["y_true"], g["y_pred"], labels=ETIQUETAS, average=tipo, zero_division=0)
        fila.update({f"precision_{tipo}": float(p_), f"recall_{tipo}": float(r_), f"f1_{tipo}": float(f_)})
    _filas.append({**fila, "support": len(g)})
report_df = pd.DataFrame(_filas)
report_df.to_csv(OUT / "classification_report_by_season.csv", index=False)
print("✔ outputs/classification_report_by_season.csv")
display(report_df.round(4))

✔ outputs/classification_report_by_season.csv


,Season,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,support
0,2010,0.4120,0.4731,0.4293,0.4886,0.6158,0.5334,380
1,2011,0.3516,0.4369,0.3833,0.4048,0.5421,0.4574,380
2,2012,0.3497,0.4200,0.3689,0.4180,0.5500,0.4622,380
3,2013,0.5274,0.4501,0.4009,0.5364,0.5526,0.4775,380
4,2014,0.5345,0.4799,0.4313,0.5439,0.5605,0.4931,380
5,2015,0.4620,0.4646,0.4237,0.4966,0.5553,0.4927,380
6,2016,0.5623,0.5076,0.4598,0.5759,0.6000,0.5301,380
7,2017,0.4780,0.4611,0.4117,0.5054,0.5579,0.4866,380
8,2018,0.4219,0.4272,0.3737,0.4392,0.4895,0.4151,380
9,2019,0.4990,0.4503,0.3988,0.5084,0.5184,0.4454,380


## 4.4 AUC y curva ROC

In [18]:
# =============================================================================
#  CELDA · CURVAS ROC Y AUC → outputs/roc_curves_by_season.json
# =============================================================================
def _curva(y, p):
    if y.min() == y.max():               # primeras jornadas: puede no haber ningún empate
        return {"fpr": [], "tpr": [], "thresholds": [], "auc": None}
    fpr, tpr, thr = roc_curve(y, p, drop_intermediate=True)
    thr = thr.copy()
    thr[0] = p.max() + 1.0               # scikit-learn pone infinito, que no es JSON válido
    return {"fpr": fpr.tolist(), "tpr": tpr.tolist(), "thresholds": thr.tolist(), "auc": float(auc(fpr, tpr))}


def bloque_roc(g):
    P = g[PROBA].to_numpy()
    Y = np.column_stack([(g["y_true"] == c).to_numpy().astype(int) for c in ETIQUETAS])
    clases = {c: _curva(Y[:, j], P[:, j]) for j, c in enumerate(ETIQUETAS)}
    areas = [v["auc"] for v in clases.values() if v["auc"] is not None]
    return {"per_class": clases, "micro": _curva(Y.ravel(), P.ravel()),
            "macro_auc": float(np.mean(areas)) if areas else None, "n_scored": len(g)}


roc = {
    "meta": {"labels": ETIQUETAS, "proba_cols": PROBA, "row_axis": "y_true (one-vs-rest)", "col_axis": "score",
             "generated_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")},
    "overall": bloque_roc(_e),
    "by_season": [{**bloque_roc(g), "Season": int(s)} for s, g in _e.groupby("Season")],
}
escribe_json(roc, OUT / "roc_curves_by_season.json")
print("✔ outputs/roc_curves_by_season.json")
display(pd.DataFrame([{"Season": b.get("Season", "todas"), **{f"AUC {c}": b["per_class"][c]["auc"] for c in ETIQUETAS},
                       "AUC macro": b["macro_auc"], "AUC micro": b["micro"]["auc"]} for b in roc["by_season"] + [roc["overall"]]]).round(4))

✔ outputs/roc_curves_by_season.json


,Season,AUC H,AUC D,AUC A,AUC macro,AUC micro
0,2010,0.7263,0.5648,0.7177,0.6696,0.7503
1,2011,0.7068,0.5608,0.7243,0.6640,0.7248
2,2012,0.7097,0.5713,0.6882,0.6564,0.7172
3,2013,0.7263,0.6083,0.7126,0.6824,0.7208
4,2014,0.7767,0.6424,0.7529,0.7240,0.7492
5,2015,0.7530,0.5752,0.7411,0.6897,0.7312
6,2016,0.7657,0.5865,0.7945,0.7156,0.7583
7,2017,0.6999,0.4832,0.7302,0.6377,0.7060
8,2018,0.6822,0.5573,0.6523,0.6306,0.6647
9,2019,0.6959,0.5466,0.7047,0.6491,0.6861


## 4.5 Beneficios

Por último, pero no por ello menos importante, vamos a estudiar la última métrica: el **ROI (Return on Investment)**.

$$
ROI = \frac{\text{Beneficio}}{\text{Inversión}}
$$

Lo que hago es simular una apuesta de un euro al resultado que predice mi modelo, en todos los partidos evaluados. Si se acierta sumamos la cuota que ofrece Bet365 menos el euro apostado, y si se falla se resta la unidad apostada. Con esto calculamos el beneficio neto y el ROI.

El detalle partido a partido queda en un **matchlog** por temporada. Los partidos por jugar también aparecen, sin apuesta ni beneficio: la aplicación los usa para saber si una jornada está completa.

In [19]:
# =============================================================================
#  CELDA · MATCHLOGS DEL MODELO → outputs/matchlogs_<temporada>.csv
# =============================================================================
def monta_matchlog(p, pred_col, proba_cols):
    m = liquida(p, pred_col).rename(columns={"Matchweek": "Matchday"})
    m["conf_maxprob"], m["entropy"], m["margin_top12"] = confianza(m[proba_cols])
    inv = 1.0 / m[["B365H", "B365D", "B365A"]].astype(float).clip(lower=1.0)
    m["overround"] = inv.sum(axis=1, skipna=False)
    m[["pH_mkt", "pD_mkt", "pA_mkt"]] = inv.div(m["overround"], axis=0).to_numpy()
    m["p_pick"] = np.select([m[pred_col] == c for c in ETIQUETAS], [m[c] for c in proba_cols], np.nan)
    b = m["odds_pick"] - 1.0
    m["ev_pick"] = m["p_pick"] * b - (1 - m["p_pick"])
    m["kelly_pick"] = (m["ev_pick"] / b).clip(0.0, 1.0)
    m = m.sort_values(["Season", "Matchday", "Date", "HomeTeam_norm", "AwayTeam_norm"], kind="mergesort").reset_index(drop=True)
    m["cum_profit_season"] = m["profit"].fillna(0.0).groupby(m["Season"]).cumsum()
    return m


COLS_LOG = ["Season", "Matchday", "Date", "HomeTeam_norm", "AwayTeam_norm", "pred_key", "pred_key_match", "y_true", "y_pred",
            "proba_H", "proba_D", "proba_A", "conf_maxprob", "entropy", "margin_top12", "B365H", "B365D", "B365A", "overround",
            "pH_mkt", "pD_mkt", "pA_mkt", "odds_pick", "p_pick", "ev_pick", "kelly_pick", "bet_placed", "correct", "profit", "cum_profit_season"]
log_modelo = monta_matchlog(preds, "y_pred", PROBA)
for s, g in log_modelo.groupby("Season"):
    g[COLS_LOG].to_csv(OUT / f"matchlogs_{int(s)}.csv", index=False)
print(f"✔ outputs/matchlogs_<temporada>.csv · {log_modelo['Season'].nunique()} temporadas, {len(log_modelo):,} partidos")

✔ outputs/matchlogs_<temporada>.csv · 17 temporadas, 6,219 partidos


## 4.6 Comparación con el modelo de Bet365

El modelo basado en las cuotas de Bet365 consiste en predecir siempre el resultado más probable según la probabilidad implícita. Se mide exactamente igual que el modelo, sobre los mismos partidos.

In [20]:
# =============================================================================
#  CELDA · MÉTRICAS DE BET365 → outputs/metrics_market_by_season.csv y
#                               outputs/metrics_market_overall.json
# =============================================================================
PROBA_MKT = ["pH_mkt_pred", "pD_mkt_pred", "pA_mkt_pred"]
metrics_market, _rk = metricas_por_temporada(preds, "y_pred_market", PROBA_MKT)
metrics_market = metrics_market[["Season", "accuracy", "logloss", "brier", "n_scored", "roi", "n_bets", "n_wins", "hit_rate",
                                 "avg_odds_win", "avg_overround"]]
metrics_market.to_csv(OUT / "metrics_market_by_season.csv", index=False)

_ak = _rk[_rk["bet_placed"] == 1]
escribe_json({
    "generated_at": datetime.now(TZ).strftime("%Y-%m-%d %H:%M:%S"),
    "model": "market_argmax(pimp1,pimpx,pimp2)",
    "overall": {
        "n_scored_total": len(_rk), "n_bets_total": len(_ak),
        "accuracy_overall": float((_rk["y_true"] == _rk["y_pred_market"]).mean()),
        "logloss_overall": log_loss_hda(_rk["y_true"], _rk[PROBA_MKT]), "brier_overall": brier_hda(_rk["y_true"], _rk[PROBA_MKT]),
        "roi_overall": float(_ak["profit"].mean()), "hit_rate_overall": float(_ak["correct"].mean()),
        "avg_overround_overall": float(metrics_market["avg_overround"].mean()),
        "avg_conf_overall": float(_rk["conf"].mean()), "avg_entropy_overall": float(_rk["ent"].mean()), "avg_margin_overall": float(_rk["mar"].mean()),
    }}, OUT / "metrics_market_overall.json")
print("✔ outputs/metrics_market_by_season.csv · outputs/metrics_market_overall.json")
display(metrics_market.round(4))

_rm = _e[PROBA].to_numpy()
print("Calidad de las probabilidades, todas las temporadas juntas (menos es mejor):")
display(pd.DataFrame({"log loss": [log_loss_hda(_e["y_true"], _rm), log_loss_hda(_rk["y_true"], _rk[PROBA_MKT])],
                      "brier": [brier_hda(_e["y_true"], _rm), brier_hda(_rk["y_true"], _rk[PROBA_MKT])]},
                     index=["modelo", "Bet365"]).round(4))

✔ outputs/metrics_market_by_season.csv · outputs/metrics_market_overall.json


,Season,accuracy,logloss,brier,n_scored,roi,n_bets,n_wins,hit_rate,avg_odds_win,avg_overround
0,2010,0.6105,0.9292,0.5457,380,0.0973,380,232,0.6105,1.7974,1.0657
1,2011,0.5368,0.9468,0.5631,380,-0.0791,380,204,0.5368,1.7154,1.0645
2,2012,0.5474,0.9428,0.5566,380,-0.0511,380,208,0.5474,1.7335,1.0637
3,2013,0.5395,0.9556,0.5648,380,-0.0922,380,205,0.5395,1.6828,1.0636
4,2014,0.5684,0.9024,0.5301,380,-0.0530,380,216,0.5684,1.6660,1.0551
5,2015,0.5474,0.9294,0.5507,380,-0.0626,380,208,0.5474,1.7126,1.0512
6,2016,0.5842,0.9027,0.5319,380,-0.0067,380,222,0.5842,1.7003,1.0508
7,2017,0.5474,0.9596,0.5705,380,-0.0360,380,208,0.5474,1.7611,1.0527
8,2018,0.4816,1.0103,0.6025,380,-0.1412,380,183,0.4816,1.7832,1.0526
9,2019,0.5184,0.9839,0.5880,380,-0.0406,380,197,0.5184,1.8506,1.0547


Calidad de las probabilidades, todas las temporadas juntas (menos es mejor):


,log loss,brier
modelo,0.9699,0.5703
Bet365,0.9544,0.5658


In [21]:
# =============================================================================
#  CELDA · MATCHLOGS DE BET365 → outputs/matchlogs_market_<temporada>.csv
# =============================================================================
COLS_LOG_MKT = ["Season", "Matchday", "Date", "HomeTeam_norm", "AwayTeam_norm", "pred_key", "y_true", "y_pred_market",
                "pH_mkt_pred", "pD_mkt_pred", "pA_mkt_pred", "conf_maxprob", "entropy", "margin_top12", "B365H", "B365D", "B365A", "overround",
                "odds_pick", "p_pick", "ev_pick", "kelly_pick", "bet_placed", "correct", "profit", "cum_profit_season"]
log_mercado = monta_matchlog(preds, "y_pred_market", PROBA_MKT)
for s, g in log_mercado.groupby("Season"):
    g[COLS_LOG_MKT].to_csv(OUT / f"matchlogs_market_{int(s)}.csv", index=False)
print(f"✔ outputs/matchlogs_market_<temporada>.csv · {log_mercado['Season'].nunique()} temporadas, {len(log_mercado):,} partidos")

✔ outputs/matchlogs_market_<temporada>.csv · 17 temporadas, 6,219 partidos


# **5. Radar plot**

Datos para la ficha de cada partido por jugar: 8 ejes del radar y las barras comparativas, en valor bruto y normalizado a [0, 1] con rangos fijos (se guardan en `schemas.json`). Un partido que ya estaba en el fichero se actualiza; los de jornadas anteriores se conservan.

In [22]:
# =============================================================================
#  CELDA · RADAR Y BARRAS → outputs/radar_prematch/
# =============================================================================
RADAR_METRICS = {                                   # eje: (local, visitante, mínimo, máximo)
    "xG7": ("home_avg_xg_last7", "away_avg_xg_last7", 0.0, 4.0),
    "OnTarget7": ("home_avg_shotsontarget_last7", "away_avg_shotsontarget_last7", 0.0, 12.0),
    "Corners7": ("home_avg_corners_last7", "away_avg_corners_last7", 0.0, 12.0),
    "Effectiveness": ("home_effectiveness", "away_effectiveness", 0.0, 1.0),
    "FormPts6": ("home_form_points_6", "away_form_points_6", 0.0, 18.0),
    "FormGD6": ("home_form_gd_6", "away_form_gd_6", -10.0, 10.0),
    "Elo": ("h_elo", "a_elo", 1450.0, 2150.0),
    "RelPerf": ("home_relative_perf", "away_relative_perf", 0.0, 2.0),
}
BARS = {                                            # barra: (local, visitante, mínimo, máximo)
    "TotalPoints": ("home_total_points_cum", "away_total_points_cum", 0.0, 114.0),
    "PointsPct": ("home_points_pct", "away_points_pct", 0.0, 1.0),
    "Position": ("home_prev_position", "away_prev_position", 1.0, 20.0),
    "GDCum": ("home_gd_cum", "away_gd_cum", -30.0, 30.0),
    "Shots7": ("home_avg_shots_last7", "away_avg_shots_last7", 0.0, 30.0),
    "Corners7_bar": ("home_avg_corners_last7", "away_avg_corners_last7", 0.0, 12.0),
    "Fouls7": ("home_avg_fouls_last7", "away_avg_fouls_last7", 5.0, 25.0),
    "Yellows7": ("home_avg_yellows_last7", "away_avg_yellows_last7", 0.0, 5.0),
    "ImpProb": ("pimp1", "pimp2", 0.0, 1.0),
}
BARS_INVERT = {"Position", "Fouls7", "Yellows7"}    # en estas, menos es mejor
NORM_VERSION = "radar_v1.0"

escribe_json({"norm_version": NORM_VERSION,
              "radar_ranges": {k: list(v[2:]) for k, v in RADAR_METRICS.items()},
              "bars_ranges": {k: list(v[2:]) for k, v in BARS.items()},
              "bars_invert": sorted(BARS_INVERT)}, RADAR / "schemas.json")

rad = df[~JUGADO].copy()
if rad.empty:
    print("No hay partidos por jugar: no se escribe nada.\nAVISO · el radar que haya en outputs/radar_prematch/ es de jornadas anteriores.")
else:
    rad["match_id"] = (rad["Season"].astype(int).astype(str) + "__" + rad["Date"].dt.strftime("%Y-%m-%d") + "__"
                       + rad["HomeTeam_norm"] + "__" + rad["AwayTeam_norm"])
    for lado in ("home", "away"):                   # % de los puntos en juego que lleva ganados
        _n = rad[f"{lado}_total_matches_prev"]
        rad[f"{lado}_points_pct"] = (rad[f"{lado}_total_points_cum"] / (3.0 * _n.where(_n > 0))).clip(0.0, 1.0)

    _faltan = sorted({c for v in (*RADAR_METRICS.values(), *BARS.values()) for c in v[:2] if c not in rad.columns})
    if _faltan:
        raise KeyError(f"columnas del radar que no están en df_final: {_faltan}")

    def _norma(c, lo, hi, invierte=False):
        x = ((rad[c] - lo) / (hi - lo + 1e-12)).clip(0, 1)
        return 1.0 - x if invierte else x

    for _, (h, a, lo, hi) in RADAR_METRICS.items():
        for c in (h, a):
            rad[f"{c}_norm"] = _norma(c, lo, hi)
    for k, (h, a, lo, hi) in BARS.items():
        for c in (h, a):
            if f"{c}_norm" not in rad.columns:      # los córners ya están en el radar, con el mismo rango
                rad[f"{c}_norm"] = _norma(c, lo, hi, k in BARS_INVERT)

    rad["has_odds"] = rad[["B365H", "B365D", "B365A"]].notna().all(axis=1)
    rad["generated_at"] = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    rad["norm_version"] = NORM_VERSION
    rad["schema_path"] = "outputs/radar_prematch/schemas.json"

    _cr = [c for v in RADAR_METRICS.values() for c in v[:2]]
    _cb = [c for v in BARS.values() for c in v[:2]]
    cols_radar = list(dict.fromkeys(
        ["Season", "Date", "Matchweek", "HomeTeam_norm", "AwayTeam_norm", "match_id", "B365H", "B365D", "B365A",
         "pimp1", "pimpx", "pimp2", "overround", "has_odds"]
        + _cr + [f"{c}_norm" for c in _cr] + _cb + ["overround"] + [f"{c}_norm" for c in _cb]
        + ["generated_at", "norm_version", "schema_path"]))

    for s, g in rad.groupby("Season"):
        ruta = RADAR / f"radar_prematch_{int(s)}.csv"
        nuevo = g[cols_radar].sort_values(["Date", "HomeTeam_norm", "AwayTeam_norm"]).copy()
        nuevo["Date"] = nuevo["Date"].dt.strftime("%Y-%m-%d")
        antes = pd.read_csv(ruta) if ruta.exists() else nuevo.iloc[:0]
        _k = ["Season", "HomeTeam_norm", "AwayTeam_norm"]     # un partido aplazado cambia de fecha, no de rivales
        _viejos = antes[~antes.set_index(_k).index.isin(nuevo.set_index(_k).index)]
        total = nuevo if _viejos.empty else pd.concat([_viejos, nuevo], ignore_index=True)
        total.to_csv(ruta, index=False, encoding="utf-8-sig")
        print(f"✔ outputs/radar_prematch/{ruta.name} · {len(nuevo)} partidos por jugar, {len(total)} en total")
    print("✔ outputs/radar_prematch/schemas.json")

✔ outputs/radar_prematch/radar_prematch_2026.csv · 10 partidos por jugar, 10 en total
✔ outputs/radar_prematch/schemas.json


# **6. Control final**

Vuelve a leer del disco lo que se acaba de escribir y comprueba que está completo y que unos ficheros cuadran con otros. Si algo no cuadra, la celda se para y dice qué.

In [23]:
# =============================================================================
#  CELDA · CONTROL FINAL
# =============================================================================
_fallos = []
def _exige(cond, mensaje):
    if not cond:
        _fallos.append(mensaje)

_temps = sorted(int(s) for s in preds["Season"].unique())
_fijos = ["metrics_main_by_season.csv", "metrics_market_by_season.csv", "metrics_market_overall.json", "confusion_matrices_by_season.json",
          "classification_report_by_season.csv", "roc_curves_by_season.json", "radar_prematch/schemas.json"]
_fijos += [f"matchlogs_{s}.csv" for s in _temps] + [f"matchlogs_market_{s}.csv" for s in _temps]
if RES_FUTURO:
    _fijos += [f"future_predictions_{c['season']}.{x}" for c in RES_FUTURO["created"] for x in ("csv", "json")]
    _fijos += [Path(RES_FUTURO["summary_path"]).name] + [f"radar_prematch/radar_prematch_{c['season']}.csv" for c in RES_FUTURO["created"]]
_no = [f for f in _fijos if not (OUT / f).exists()]
_exige(not _no, f"faltan ficheros en outputs/: {_no}")

if not _no:
    _mm = pd.read_csv(OUT / "metrics_main_by_season.csv").set_index("Season")
    _mk = pd.read_csv(OUT / "metrics_market_by_season.csv").set_index("Season")
    _cm = json.loads((OUT / "confusion_matrices_by_season.json").read_text(encoding="utf-8"))
    _cr = pd.read_csv(OUT / "classification_report_by_season.csv").set_index("Season")
    _lm = pd.concat([pd.read_csv(OUT / f"matchlogs_{s}.csv") for s in _temps], ignore_index=True)
    _lk = pd.concat([pd.read_csv(OUT / f"matchlogs_market_{s}.csv") for s in _temps], ignore_index=True)

    _exige(set(_lm["pred_key"]) == set(_lk["pred_key"]), "los matchlogs del modelo y de Bet365 no tienen los mismos partidos")
    for _nom, _met, _log in (("modelo", _mm, _lm), ("Bet365", _mk, _lk)):
        _a = _log[_log["bet_placed"] == 1].groupby("Season")[["profit", "correct"]].mean().reindex(_met.index)
        _exige(np.allclose(_a["profit"], _met["roi"], atol=1e-9, equal_nan=True), f"el ROI de los matchlogs y el de las métricas no coinciden ({_nom})")
        _exige(np.allclose(_a["correct"], _met["hit_rate"], atol=1e-9, equal_nan=True), f"el acierto de los matchlogs y el de las métricas no coinciden ({_nom})")
        _n = _log[_log["y_true"].isin(ETIQUETAS)].groupby("Season").size().reindex(_met.index)
        _ll = float((_met["logloss"] * _n).sum() / _n.sum())
        _exige(_ll < np.log(3), f"log loss {_ll:.3f} ({_nom}), peor que repartir la probabilidad a partes iguales (ln 3 = {np.log(3):.3f})")
    _exige(all(np.isclose(np.trace(b["matrix"]) / b["n_scored"], _mm.loc[b["Season"], "accuracy"]) for b in _cm["by_season"]),
           "la matriz de confusión no cuadra con el acierto")
    _exige(int(_cr["support"].sum()) == _cm["overall"]["n_scored"] == int(JUG.sum()), "el número de partidos evaluados no coincide entre ficheros")
    _exige(np.allclose(_lm[["proba_H", "proba_D", "proba_A"]].sum(axis=1), 1.0, atol=1e-9), "hay probabilidades que no suman 1 en los matchlogs")

    if RES_FUTURO:
        _sm = json.loads((OUT / Path(RES_FUTURO["summary_path"]).name).read_text(encoding="utf-8"))
        _exige(_sm["model"]["features"] == list(FEATURES_FINAL), "las predicciones de la jornada no se hicieron con FEATURES_FINAL")
        for c in RES_FUTURO["created"]:
            _fp = pd.read_csv(OUT / f"future_predictions_{c['season']}.csv").set_index("pred_key")
            _fl = _lm[~_lm["y_true"].isin(ETIQUETAS) & (_lm["Season"] == c["season"])].set_index("pred_key").reindex(_fp.index)
            _exige(np.allclose(_fp[["pH_pred", "pD_pred", "pA_pred"]], _fl[["proba_H", "proba_D", "proba_A"]], atol=1e-9),
                   "la predicción de la jornada y la del matchlog no coinciden")
            _rd = pd.read_csv(OUT / "radar_prematch" / f"radar_prematch_{c['season']}.csv")
            _exige(set(zip(_fp["HomeTeam_norm"], _fp["AwayTeam_norm"])) <= set(zip(_rd["HomeTeam_norm"], _rd["AwayTeam_norm"])),
                   "hay partidos de la jornada sin ficha de radar")

if _fallos:
    raise AssertionError("CONTROL FINAL · no cuadra:\n  - " + "\n  - ".join(_fallos))

print(f"CONTROL FINAL correcto: {len(_fijos)} ficheros presentes y coherentes entre sí.")
print(f"Variables: {list(FEATURES_FINAL)}")
print(f"Acierto {100 * RES_MODELO['acierto']:.2f} % (Bet365 {100 * RES_BET365['acierto']:.2f} %) · "
      f"ROI {100 * RES_MODELO['roi']:+.2f} % (Bet365 {100 * RES_BET365['roi']:+.2f} %)")

# Resúmenes de jornadas anteriores que se hicieron con otras variables: se conservan, pero conviene saberlo
_otros = []
for _f in sorted(OUT.glob("future_predictions_summary_*.json")):
    if json.loads(_f.read_text(encoding="utf-8")).get("model", {}).get("features") != list(FEATURES_FINAL):
        _otros.append(_f.name)
if _otros:
    print(f"\nAVISO · {len(_otros)} resúmenes de predicción anteriores usaron otras variables: {_otros}")

CONTROL FINAL correcto: 45 ficheros presentes y coherentes entre sí.
Variables: ['pimp1', 'pimpx', 'home_form_gd_6', 'away_relative_perf', 'form_points_6_diff', 'away_avg_big_ch_against_last7', 'home_avg_box_touches_last7', 'home_avg_sofa_rating_last7']
Acierto 55.19 % (Bet365 54.58 %) · ROI +0.01 % (Bet365 -2.77 %)
